# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [19]:
!git clone https://github.com/ainasarfaraz343-a11y/flyrank-internship.git

Cloning into 'flyrank-internship'...
remote: Enumerating objects: 128, done.
remote: Counting objects: 100% (128/128), done.
remote: Compressing objects: 100% (99/99), done.
remote: Total 128 (delta 39), reused 78 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (128/128), 1.86 MiB | 8.78 MiB/s, done.
Resolving deltas: 100% (39/39), done.


In [20]:
%cd flyrank-internship

/content/flyrank-internship/flyrank-internship


In [21]:
import os
print(os.getcwd())
!ls data/raw/

/content/flyrank-internship/flyrank-internship
content_refresh_anonymized.csv


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [22]:
import pandas as pd

df = pd.read_csv('data/raw/content_refresh_anonymized.csv')
print(df.shape)

(30000, 44)


## 1. Signal checks & rule

**Signal 1: Staleness (freshness_tier) vs decline rate**
Ordering by staleness (0-30 → 31-90 → 91-180 → 181+), decline rate rises from
0.511 → 0.589 → 0.611 across the bulk of the data (0-30 and 91-180 together
hold ~98% of rows), then drops to 0.471 at the 181+ tier — but that tier has
only n=174 rows, too small to trust. **Verdict: MIXED** — the core relationship
(more stale → more decline) holds where the data is dense, but breaks at the
sparse tail.

**Signal 2: Volume (impression_tier) vs decline rate**
Ordering by volume (low → moderate → good → excellent), decline rate goes
0.454 → 0.615 → 0.586 → 0.462 — a hump shape, not a monotonic trend. Every
bucket has n > 1,000, so this isn't a small-sample fluke — volume alone does
not cleanly predict decline. **Verdict: MIXED**

**Rule:** A page is worth reviewing if it has gone stale (days_since_last_update
>= 90) AND still has real impression volume (impressions_90d >= 300). Neither
signal cleanly predicts decline on its own — that's expected, since the rule
isn't trying to predict decline, it's flagging a specific *combination* worth a
human look: content that's aging but hasn't lost visibility yet, so a refresh
has something to work with.

**Reason codes:** STALE_HIGH_VOLUME, STALE_LOW_VOLUME, FRESH_NO_ACTION

In [23]:
df['is_declining'] = (df['trend_direction'] == 'down').astype(int)

# Signal 1: Staleness vs decline rate
sig1 = df.groupby('freshness_tier').agg(
    n=('is_declining', 'size'),
    decline_rate=('is_declining', 'mean')
).round(3)
print("Signal 1 — Staleness (freshness_tier) vs decline rate")
print(sig1)
print()

# Signal 2: Volume vs decline rate
sig2 = df.groupby('impression_tier').agg(
    n=('is_declining', 'size'),
    decline_rate=('is_declining', 'mean')
).round(3)
print("Signal 2 — Volume (impression_tier) vs decline rate")
print(sig2)

Signal 1 — Staleness (freshness_tier) vs decline rate
                    n  decline_rate
freshness_tier                     
0-30            20480         0.511
181+              174         0.471
31-90             175         0.589
91-180           9171         0.611

Signal 2 — Volume (impression_tier) vs decline rate
                     n  decline_rate
impression_tier                     
excellent         1078         0.462
good              7205         0.586
low              11248         0.454
moderate         10469         0.615


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

## 2. Ranked queue
Score = is_stale × has_volume × impressions_90d. Since is_stale/has_volume are
binary gates, the top of the queue ranks purely by impressions among qualifying
pages — visible in the output below, where every top-10 row happens to sit at
the same days_since_last_update (104). Queue written to
work/outputs/baseline_action_score.csv.

In [24]:
import os

# Score: transparent, readable — no fitted weights
df['is_stale'] = (df['days_since_last_update'] >= 90).astype(int)
df['has_volume'] = (df['impressions_90d'] >= 300).astype(int)   # "moderate" tier cutoff se

df['score'] = df['is_stale'] * df['has_volume'] * df['impressions_90d']

def reason_code(row):
    if row['is_stale'] and row['has_volume']:
        return 'STALE_HIGH_VOLUME'
    elif row['is_stale'] and not row['has_volume']:
        return 'STALE_LOW_VOLUME'
    else:
        return 'FRESH_NO_ACTION'

df['reason_code'] = df.apply(reason_code, axis=1)
df['action'] = df['reason_code'].apply(lambda r: 'REVIEW_REFRESH' if r == 'STALE_HIGH_VOLUME' else 'NO_ACTION')

queue = df.sort_values('score', ascending=False)[
    ['content_id', 'client_id', 'score', 'reason_code', 'action',
     'days_since_last_update', 'impressions_90d', 'avg_position', 'ctr']
]

os.makedirs('work/outputs', exist_ok=True)
queue.to_csv('work/outputs/baseline_action_score.csv', index=False)
print(queue.head(10))

                 content_id          client_id   score        reason_code  \
6653   content_5fe46e04994d  client_4e07408562  517715  STALE_HIGH_VOLUME   
29400  content_2dba2b1f9536  client_6208ef0f77  443434  STALE_HIGH_VOLUME   
13537  content_2c2606c5d176  client_19581e27de  347399  STALE_HIGH_VOLUME   
26531  content_cb112fce36be  client_19581e27de  309910  STALE_HIGH_VOLUME   
21565  content_9532f197bbc8  client_4e07408562  309192  STALE_HIGH_VOLUME   
3394   content_36ff89c8214e  client_19581e27de  295097  STALE_HIGH_VOLUME   
26798  content_b28d1efd668f  client_6208ef0f77  286608  STALE_HIGH_VOLUME   
23767  content_813e88069237  client_6208ef0f77  233561  STALE_HIGH_VOLUME   
26255  content_c21024970297  client_19581e27de  211366  STALE_HIGH_VOLUME   
7445   content_c8e9d6ab9013  client_19581e27de  208678  STALE_HIGH_VOLUME   

               action  days_since_last_update  impressions_90d  avg_position  \
6653   REVIEW_REFRESH                     104           517715          

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

## 3. Top-10 review

1. 517,715 impressions, position 4.2, ctr 0.14 — REVIEW_REFRESH: top rank already,
   good position but low CTR for that position — refresh could lift CTR.
   **Wrong if:** low CTR is due to a non-clickable snippet type (e.g. featured
   snippet stealing clicks), not stale content.

2. 443,434 impressions, position 27.9, ctr 0.21 — REVIEW_REFRESH: page-3 position
   with real impression volume — classic "visible but buried" case.
   **Wrong if:** position 27.9 reflects ranking volatility/multiple keywords, not
   a real stuck position — a single-day check could be misleading.

3. 347,399 impressions, position 4.2, ctr 0.53 — REVIEW_REFRESH: strong position
   AND decent CTR already — the rule flagged it purely on volume+staleness.
   **Wrong if:** this page is already performing near its ceiling and doesn't
   need a refresh at all — a weak pick.

4. 309,910 impressions, position 5.6, ctr 0.16 — REVIEW_REFRESH: good position,
   underperforming CTR — refresh (title/meta) is a reasonable bet.
   **Wrong if:** CTR is capped by intent mismatch (query doesn't match content),
   which a refresh won't fix.

5. 309,192 impressions, position 2.0, ctr 0.87 — REVIEW_REFRESH: top position,
   best CTR in this batch — barely needs refreshing.
   **Wrong if:** flagged only because of the volume gate; a stronger rule would
   exclude already-high performers like this.

6. 295,097 impressions, position 7.3, ctr 0.05 — REVIEW_REFRESH: decent position
   but very low CTR — title/snippet likely the problem.
   **Wrong if:** low CTR is seasonal/topic-driven rather than staleness-driven.

7. 286,608 impressions, position 26.2, ctr 0.06 — REVIEW_REFRESH: buried position,
   low CTR — strong refresh candidate on paper.
   **Wrong if:** the page targets a highly competitive keyword where refresh
   alone won't move position 26 → page 1.

8. 233,561 impressions, position 26.2, ctr 0.06 — REVIEW_REFRESH: same profile
   as #7 — buried, low CTR.
   **Wrong if:** duplicate/near-duplicate content competing with another owned
   page for the same query (cannibalization, not staleness).

9. 211,366 impressions, position 5.1, ctr 0.41 — REVIEW_REFRESH: solid position
   and CTR already — another borderline/weak flag from the volume gate.
   **Wrong if:** already near-optimal, refresh effort better spent elsewhere.

10. 208,678 impressions, position 9.7, ctr 0.00 — REVIEW_REFRESH: real impression
    volume but **zero clicks** — could mean broken link, wrong intent match, or
    tracking issue, not simply "stale."
    **Wrong if:** the 0.00 CTR is a data/tracking artifact rather than a genuine
    content problem — refresh wouldn't fix a tracking bug.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*


Weak picks: rows #3, #5, and #9 already have strong position and CTR
(ctr 0.53, 0.87, 0.41) — the rule flagged them purely because they cleared the
staleness/volume gates, not because they show real performance decline. A
sharper rule would also gate on CTR-vs-position underperformance, not just
staleness + volume.

Leakage check: trend_direction and trend_pct were used ONLY in Section 1 to
verify signal quality (EDA) — neither appears in the score, reason_code, or
action logic in Section 2. No future-window or label-derived columns were used
as rule inputs.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.